# Modelo para escolha de plano telefônico

## Descrição do projeto

A operadora de celular Megaline está insatisfeita com o fato de muitos de seus clientes estarem usando planos antigos. Ela quer desenvolver um modelo que possa analisar o comportamento do cliente e recomendar um dos planos mais recentes da Megaline: Smart ou Ultra.

Você tem acesso a dados de comportamento dos clientes que já mudaram para os novos planos (do projeto do curso de [Análise de dados estatísticos](https://github.com/peperrone/mbads-sprint5-AED)). Para essa tarefa de classificação, você precisa desenvolver um modelo que escolhe o plano certo. Como você já executou a etapa de pré-processamento de dados, pode ir direto para a criação do modelo.

Desenvolva um modelo com a maior acurácia possível. Neste projeto, o limite para acurácia é 0,75. Verifique a acurácia usando o conjunto de dados de teste.

### Instruções do projeto
- <input type="checkbox"> Abra e examine o arquivo de dados.
- <input type="checkbox"> Divida os dados de origem em um conjunto de treinamento, um conjunto de validação e um conjunto de teste.
- <input type="checkbox"> Investigue a qualidade de diferentes modelos alterando hiperparâmetros. Descreva brevemente os resultados do estudo.
- <input type="checkbox"> Verifique a qualidade do modelo usando o conjunto de teste.

**Tarefa adicional:** 
- <input type="checkbox"> Tirar a prova real do modelo. Esses dados são mais complexos do que os que você está acostumado a trabalhar, então não será uma tarefa fácil. Vamos dar uma olhada mais de perto mais tarde

## Inicialização

In [17]:
#importando bibliotecas
import numpy as np
import pandas as pd

In [5]:
# Importando o dataset
try:
    df = pd.read_csv('datasets/users_behavior.csv') # caminho local no windows
except FileNotFoundError:
    df = pd.read_csv('/datasets/users_behavior.csv') # caminho no ambiente da tripleten

## Examinando os dados

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [7]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [8]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [47]:
# Verificando a distribuição da variável alvo
print(df['is_ultra'].value_counts())

is_ultra
0    2229
1     985
Name: count, dtype: int64


In [39]:
# Dividindo o dataset nas nossas duas categorias: ultra e surf (não ultra)
df_ultra = df[df['is_ultra'] == 1]
df_surf = df[df['is_ultra'] == 0]

In [40]:
df_ultra.describe()

,calls,minutes,messages,mb_used,is_ultra
count,985.000000,985.000000,985.000000,985.000000,985.0
mean,73.392893,511.224569,49.363452,19468.823228,1.0
std,43.916853,308.031100,47.804457,10087.178654,0.0
min,0.000000,0.000000,0.000000,0.000000,1.0
25%,41.000000,276.030000,6.000000,11770.280000,1.0
50%,74.000000,502.550000,38.000000,19308.010000,1.0
75%,104.000000,730.050000,79.000000,26837.720000,1.0
max,244.000000,1632.060000,224.000000,49745.730000,1.0


In [41]:
df_surf.describe()

,calls,minutes,messages,mb_used,is_ultra
count,2229.000000,2229.000000,2229.000000,2229.000000,2229.0
mean,58.463437,405.942952,33.384029,16208.466949,0.0
std,25.939858,184.512604,28.227876,5870.498853,0.0
min,0.000000,0.000000,0.000000,0.000000,0.0
25%,40.000000,274.230000,10.000000,12643.050000,0.0
50%,60.000000,410.560000,28.000000,16506.930000,0.0
75%,76.000000,529.510000,51.000000,20043.060000,0.0
max,198.000000,1390.220000,143.000000,38552.620000,0.0


## Preparação dos conjuntos de treinamento, validação e teste

In [14]:
# Importando a função para dividir o dataset
from sklearn.model_selection import train_test_split

In [15]:
# Dividindo o dataset em target e features
features = df.drop('is_ultra', axis=1) 
target = df['is_ultra']

# Dividindo o dataset em treino e o subconjunto de teste+validação
features_train, features_test_val, target_train, target_test_val = train_test_split(features, target, test_size=0.4, random_state=42)

In [16]:
## Dividindo o subcojunto em teste e validação
# 50% do subconjunto de teste+validação para cada um (20% do total para cada um)
features_val, features_test, target_val, target_test = train_test_split(features_test_val, target_test_val, test_size=0.5, random_state=42) 

## Testando modelos

In [24]:
# Importando modelos de classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Importando as funções para avaliar os modelos
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Regressão logística

In [ ]:
# Criando uma instância do modelo de regressão logística
log_model = LogisticRegression(random_state=42, solver='liblinear') # solver 'liblinear' é recomendado para conjuntos de dados pequenos

# Treinando o modelo de regressão logística
log_model.fit(features_train, target_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [ ]:
# Valiando o modelo de regressão logística
log_predictions = log_model.predict(features_val)

log_accuracy = accuracy_score(target_val, log_predictions)
print(f"Acurácia do modelo de Regressão Logística: {log_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, log_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, log_predictions))

Acurácia do modelo de Regressão Logística: 0.72

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.72      0.98      0.83       446
           1       0.79      0.14      0.23       197

    accuracy                           0.72       643
   macro avg       0.76      0.56      0.53       643
weighted avg       0.74      0.72      0.65       643


Matriz de Confusão:
 [[439   7]
 [170  27]]


In [ ]:
# Fazendo testes com outros solvers do modelo de regressão logística
solvers = ['lbfgs', 'liblinear', 'newton-cg'] # excluindo sag e saga por serem mais lentos em conjuntos de dados pequenos e liblinear por já ter sido testado
for solver in solvers:
    log_model = LogisticRegression(random_state=42, solver=solver)
    log_model.fit(features_train, target_train)
    predictions = log_model.predict(features_val)
    accuracy = accuracy_score(target_val, predictions)
    print(f"Solver: {solver}, Acurácia: {accuracy:.2f}")
    print("\nRelatório de Classificação:\n", classification_report(target_val, predictions))
    print("\nMatriz de Confusão:\n", confusion_matrix(target_val, predictions))
    print()



Solver: lbfgs, Acurácia: 0.74

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.74      0.97      0.84       446
           1       0.78      0.21      0.33       197

    accuracy                           0.74       643
   macro avg       0.76      0.59      0.59       643
weighted avg       0.75      0.74      0.68       643


Matriz de Confusão:
 [[434  12]
 [155  42]]

Solver: liblinear, Acurácia: 0.72

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.72      0.98      0.83       446
           1       0.79      0.14      0.23       197

    accuracy                           0.72       643
   macro avg       0.76      0.56      0.53       643
weighted avg       0.74      0.72      0.65       643


Matriz de Confusão:
 [[439   7]
 [170  27]]

Solver: newton-cg, Acurácia: 0.74

Relatório de Classificação:
               precision    recall  f1-score   support

           

In [59]:
# Seguindo com lbfgs, que teve a melhor acurácia e testando hiperparâmetros do modelo de regressão logística
log_model = LogisticRegression(random_state=42, solver='lbfgs', class_weight='balanced') # class_weight='balanced' é recomendado para lidar com classes desbalanceadas
log_model.fit(features_train, target_train)
log_predictions = log_model.predict(features_val)
log_accuracy = accuracy_score(target_val, log_predictions)
print(f"Acurácia do modelo de Regressão Logística com class_weight='balanced': {log_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, log_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, log_predictions))

Acurácia do modelo de Regressão Logística com class_weight='balanced': 0.63

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.78      0.64      0.71       446
           1       0.42      0.60      0.50       197

    accuracy                           0.63       643
   macro avg       0.60      0.62      0.60       643
weighted avg       0.67      0.63      0.64       643


Matriz de Confusão:
 [[286 160]
 [ 79 118]]


In [71]:
# Testando combinações de hiperparâmetros do modelo de regressão logística usando GridSearchCV
from sklearn.model_selection import GridSearchCV

# Definindo o grid de hiperparâmetros para o modelo de regressão logística
param_grid = {
    'solver': ['lbfgs', 'liblinear', 'newton-cg'],
    'class_weight': ['balanced'],
    'C': [0.01, 0.1, 1, 10], # C é o inverso da regularização, valores menores significam mais regularização
    'max_iter': [100, 200, 1000, 5000], # número máximo de iterações para o algoritmo de otimização
}

In [72]:
# Reiniciando o modelo de regressão logística para usar no GridSearchCV
log_model = LogisticRegression(random_state=42)

# Criando o GridSearchCV para o modelo de regressão logística
gs_log_model = GridSearchCV(log_model, param_grid=param_grid, cv=5, scoring='accuracy')

In [74]:
gs_log_model.fit(features_train, target_train)

e:\DataScience\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\DataScience\anaconda3\Lib\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
e:\DataScience\anaconda3\Lib\site-packages\sklearn\utils\optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(
e:\DataScience\anaconda3\Lib\site-pack

,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': ['balanced'], 'max_iter': [100, 200, ...], 'solver': ['lbfgs', 'liblinear', ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [75]:
print(f"Melhores hiperparâmetros encontrados: {gs_log_model.best_params_}")
print(f"Melhor acurácia encontrada: {gs_log_model.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'C': 0.1, 'class_weight': 'balanced', 'max_iter': 100, 'solver': 'lbfgs'}
Melhor acurácia encontrada: 0.64


### Árvore de decisão